# Push-T in-context goal inference: two learner seeds in parallel (Jimmy)

A GPT-2-style learner (12 layers, width 256, 8 heads; `PushT_ICL_Transformer_Spec.md`) sees demonstrations of a frozen
teacher pushing to a **hidden** goal and outputs the teacher's action at a new state. Goal coordinates are removed from every
input (28 of the 31 numbers). Trained on Q1–Q3 goals, selected on separate Q1–Q3 goals, tested frozen on unseen Q1–Q3 goals
and on Q4 goals (`CODEX_CONTEXT.md`). Teachers in this notebook: **ppo**.

| Setting | Value |
|---|---|
| Data | Nathan's conventions: goals train 2,048 / val 128 / test_in 128 (Q1–Q3) / test_q4 128 (Q4), 10 noise-free episodes per goal and teacher, same goal/demo/eval seeds, same file format |
| Data cache | if a previous run's output is attached as input, its demos are reused; otherwise collected here (few minutes) |
| Prompt | k ∈ {0, 1, 2, 4, 8} successful demos (whole episode, every 4th step, ending at step 100) + one query step from another episode |
| Training | **two seeds at once, one per T4**; fp16 autocast, Adam, batch 64, grad clip 1.0; settings in the first cell |
| Selection | lowest val action MSE with correct demos (k = 1–8, averaged over teachers) |
| Evaluation | per teacher: correct-goal / wrong-goal / no demos and the teacher itself (true goal); 4 starts per goal, identical across conditions |

The PPO teacher is embedded as its actor (`state_rl/teachers/ppo_vargoal_seed1_50M.pt`, sha256 430807142173472f): Kaggle caps imports at 1 MB.

**Kaggle:** Settings → Accelerator → **GPU T4 x2**, Internet on. Save Version → **Save & Run All (Commit)**.

In [ ]:
from pathlib import Path
import os, sys, json, subprocess, datetime, shlex, shutil, base64, hashlib

MANISKILL_COMMIT = "a4a4f9272ad64b1564035874b605ceb687b63ed8"
BASE = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("/content")
MS_ROOT = Path("/tmp/ManiSkill")
WORK = Path("/tmp/pusht_icl"); WORK.mkdir(parents=True, exist_ok=True)
SESSION = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_icl_two_seed"
OUTPUT_ROOT = BASE / "state_icl" / SESSION
OUTPUT_ROOT.mkdir(parents=True, exist_ok=False)
LOG = OUTPUT_ROOT / "notebook.log"

TEACHERS = ['ppo']
SEEDS = [0, 1]             # one per GPU
TRAIN_STEPS = 50_000
WARMUP, LR_MIN_FRAC = 0, 1.0   # constant lr; Nathan's schedule is WARMUP=1000, LR_MIN_FRAC=0.1
MAX_MINUTES = 600          # stop training early if it is slower than expected (Kaggle limit: 12 h)
EVAL_EVERY = 2000
subprocess.run(["nvidia-smi"], check=True)
print("Output:", OUTPUT_ROOT)


def run(command, **kwargs):
    command = list(map(str, command))
    print("$", shlex.join(command), flush=True)
    tail = []
    with LOG.open("a") as log:
        log.write("\n$ " + shlex.join(command) + "\n")
        with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
                              **kwargs) as p:
            for line in p.stdout:
                print(line, end="", flush=True); log.write(line); tail = (tail + [line.rstrip()])[-40:]
            code = p.wait()
    if code:
        raise RuntimeError(f"Command failed (exit {code})\n" + "\n".join(tail))


def run_parallel(jobs):
    # jobs: {label: (command, gpu)}. Runs them at once, one GPU each; prints their output prefixed by label.
    import time as _t
    procs = {}
    for label, (command, gpu) in jobs.items():
        command = list(map(str, command))
        print(f"[{label}] $", shlex.join(command), flush=True)
        log = open(OUTPUT_ROOT / f"{label}.log", "w")
        procs[label] = (subprocess.Popen(command, stdout=log, stderr=subprocess.STDOUT, text=True,
                                         env={**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu)}), log, [0])
    while procs:
        _t.sleep(30)
        for label, (p, log, pos) in list(procs.items()):
            log.flush()
            lines = (OUTPUT_ROOT / f"{label}.log").read_text().splitlines()
            for line in lines[pos[0]:]:
                if "warn" not in line.lower():   # skip library warnings
                    print(f"[{label}] {line}", flush=True)
            pos[0] = len(lines)
            if p.poll() is not None:
                log.close(); del procs[label]
                if p.returncode:
                    raise RuntimeError(f"{label} failed (exit {p.returncode}); see {label}.log")

## 1. Install ManiSkill v3.0.1 (Python 3.11) and Transformers

In [ ]:
run(["apt-get", "update", "-qq"])
run(["apt-get", "install", "-y", "-qq", "libvulkan1", "libvulkan-dev", "vulkan-tools", "libglvnd-dev"])
if not (MS_ROOT / ".git").exists():
    run(["git", "clone", "https://github.com/mani-skill/ManiSkill.git", MS_ROOT])
run(["git", "-C", MS_ROOT, "checkout", "--detach", MANISKILL_COMMIT])
run([sys.executable, "-m", "pip", "install", "-q", "uv==0.9.11"])
ENV = WORK / "python311"; PY = ENV / "bin/python"
if not PY.exists():
    run([sys.executable, "-m", "uv", "venv", "--python", "3.11", "--seed", ENV])
constraints = WORK / "constraints.txt"
constraints.write_text("numpy==1.26.4\nopencv-python==4.11.0.86\nsapien==3.0.3\ntorch==2.6.0\n")
run([PY, "-m", "pip", "install", "-q", "torch==2.6.0", "--index-url", "https://download.pytorch.org/whl/cu124"])
run([PY, "-m", "pip", "install", "-q", "--constraint", constraints, "-e", MS_ROOT,
     "numpy", "sapien", "h5py", "tyro", "transformers==4.46.3"])
for filename, library, vulkan in [("/usr/share/vulkan/icd.d/nvidia_icd.json", "libGLX_nvidia.so.0", True),
                                  ("/usr/share/glvnd/egl_vendor.d/10_nvidia.json", "libEGL_nvidia.so.0", False)]:
    path = Path(filename)
    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        icd = {"library_path": library}
        if vulkan:
            icd["api_version"] = "1.2.155"
        path.write_text(json.dumps({"file_format_version": "1.0.0", "ICD": icd}))
os.environ.update(VK_ICD_FILENAMES="/usr/share/vulkan/icd.d/nvidia_icd.json", MS_SKIP_ASSET_DOWNLOAD_PROMPT="1",
                  PYTHONUNBUFFERED="1", CUDA_VISIBLE_DEVICES="0")
run([PY, "-c", "import torch, transformers, mani_skill; print('torch', torch.__version__, 'transformers', "
     "transformers.__version__, 'mani_skill', mani_skill.__version__); assert torch.cuda.is_available()"])
(OUTPUT_ROOT / "pip-freeze.txt").write_text(subprocess.check_output([str(PY), "-m", "pip", "freeze"], text=True))

## 2. Write the code and the frozen PPO teacher

Generated by `state_icl/build_notebook.py`. Do not edit here.

In [ ]:
SRC = WORK / "repo"
for d in ("state_rl/assets", "state_rl/teachers", "state_icl/teachers"):
    (SRC / d).mkdir(parents=True, exist_ok=True)

In [ ]:
%%writefile /tmp/pusht_icl/repo/state_rl/pusht_vargoal.py
"""Variable-goal Push-T: ManiSkill PushT-v1 with a different target position per episode.

Registers ``PushTVarGoal-v1``. Import this module before ``gym.make``.

What changes relative to PushT-v1 (ManiSkill 3.0.1, commit a4a4f92):
  * Goal T position is sampled per episode from four quadrants around ``QUAD_ORIGIN``.
    Goal orientation stays fixed at PushT-v1's 5*pi/3.
  * The success check (``pseudo_render_intersection``) uses a goal transform per
    environment, recomputed at every reset. In PushT-v1 it is one fixed matrix, so a
    moved goal would silently be scored against the old target.
  * The T block starts anywhere in ``TEE_BOX`` (independent of the goal quadrant), with
    its distance to the goal limited to ``START_GOAL_DIST``.
  * The stick tip starts at a random point in ``TIP_BOX``, drawn from a precomputed
    library of arm poses (``assets/tip_start_qpos.npz``, built by make_tip_starts.py),
    at least ``TIP_TEE_MIN`` from the block.

What does NOT change: observation (31 numbers, same order; ``extra/goal_pos`` now
varies), action space and controller, 100-step episodes, the dense reward (it already
reads the goal T's live pose), and the success rule (overlap >= 0.90).

Quadrants, with dx = x - QUAD_ORIGIN[0] (positive = away from the robot base) and y:
  Q1: dx > 0, y > 0    Q2: dx < 0, y > 0    Q3: dx < 0, y < 0    Q4: dx > 0, y < 0
A band of ``AXIS_GAP`` around each axis is excluded so neighbouring quadrants do not
share near-identical goals.

Usage:
  env = gym.make("PushTVarGoal-v1", obs_mode="state", control_mode="pd_joint_delta_pos",
                 goal_quadrants=(1, 2, 3, 4))
  env.reset(seed=0)                                    # random goals in those quadrants
  env.reset(options=dict(goal_xy=torch.tensor([[-0.2, 0.1]])))  # fixed goal(s), [b, 2]
  env.unwrapped.goal_quadrant                          # [num_envs] quadrant of current goal
"""
from pathlib import Path

import numpy as np
import torch

from mani_skill.envs.tasks.tabletop.push_t import PushTEnv
from mani_skill.utils.registration import register_env
from mani_skill.utils.structs.pose import Pose

# ---- Workspace (world frame, metres). Robot base is at x = -0.615. ----------------
# Measured stick-tip reach at table height: x in about [-0.40, 0.05], y in about
# [-0.45, 0.45] (see README). Regions below keep >= ~0.1 m margin inside that.
QUAD_ORIGIN = (-0.156, 0.0)          # PushT-v1's goal x; centred in y
GOAL_HALF = (0.10, 0.20)             # goals in origin +- these
AXIS_GAP = (0.02, 0.03)              # excluded band around each quadrant axis
TEE_BOX = ((-0.30, -0.02), (-0.28, 0.28))   # T centre-of-mass start region
TIP_BOX = ((-0.36, 0.00), (-0.30, 0.30))    # stick-tip start region
START_GOAL_DIST = (0.08, 0.30)       # allowed T-start-to-goal distance
TIP_TEE_MIN = 0.15                   # stick tip starts at least this far from the T
N_CANDIDATES = 64                    # rejection-sampling candidates per reset

_QUAD_SIGNS = {1: (1.0, 1.0), 2: (-1.0, 1.0), 3: (-1.0, -1.0), 4: (1.0, -1.0)}
_DEFAULT_TIP_FILE = Path(__file__).parent / "assets" / "tip_start_qpos.npz"


def quadrant_of(goal_xy: torch.Tensor) -> torch.Tensor:
    """Quadrant (1-4) of goal positions [b, 2]."""
    dx = goal_xy[:, 0] - QUAD_ORIGIN[0]
    dy = goal_xy[:, 1] - QUAD_ORIGIN[1]
    q = torch.full_like(dx, 0, dtype=torch.long)
    q[(dx > 0) & (dy > 0)] = 1
    q[(dx < 0) & (dy > 0)] = 2
    q[(dx < 0) & (dy < 0)] = 3
    q[(dx > 0) & (dy < 0)] = 4
    return q


def sample_goals(b: int, quadrants, device=None) -> torch.Tensor:
    """Uniform goal positions [b, 2] in the given quadrants (each quadrant equally likely)."""
    quads = torch.tensor(list(quadrants), device=device)
    pick = quads[torch.randint(len(quads), (b,), device=device)]
    signs = torch.tensor([_QUAD_SIGNS[int(k)] for k in range(1, 5)], device=device)[pick - 1]
    lo = torch.tensor(AXIS_GAP, device=device)
    hi = torch.tensor(GOAL_HALF, device=device)
    mag = lo + torch.rand(b, 2, device=device) * (hi - lo)
    return torch.tensor(QUAD_ORIGIN, device=device) + signs * mag


def _uniform_box(shape, box, device):
    lo = torch.tensor([box[0][0], box[1][0]], device=device)
    hi = torch.tensor([box[0][1], box[1][1]], device=device)
    return lo + torch.rand(*shape, 2, device=device) * (hi - lo)


def _first_valid(valid: torch.Tensor) -> torch.Tensor:
    """Index of the first True per row of [b, K]; 0 if none (vanishingly rare with K=64)."""
    return valid.float().argmax(dim=1)


@register_env("PushTVarGoal-v1", max_episode_steps=100)
class PushTVarGoalEnv(PushTEnv):
    def __init__(self, *args, goal_quadrants=(1, 2, 3, 4), tip_start_file=None, **kwargs):
        assert set(goal_quadrants) <= {1, 2, 3, 4} and len(goal_quadrants) > 0
        self.goal_quadrants = tuple(int(q) for q in goal_quadrants)
        lib = np.load(tip_start_file or _DEFAULT_TIP_FILE)
        self._tip_qpos_np = lib["qpos"]
        self._tip_xy_np = lib["tip_xy"]
        super().__init__(*args, **kwargs)

    def _load_scene(self, options: dict):
        super()._load_scene(options)
        # One goal transform per environment instead of PushT-v1's single matrix.
        self.world_to_goal_trans = self.world_to_goal_trans.unsqueeze(0).repeat(self.num_envs, 1, 1)
        self.goal_quadrant = torch.zeros(self.num_envs, dtype=torch.long, device=self.device)
        self.tip_qpos_lib = torch.as_tensor(self._tip_qpos_np, device=self.device)
        self.tip_xy_lib = torch.as_tensor(self._tip_xy_np, device=self.device)

    def _goal_world_to_goal(self, goal_xy: torch.Tensor) -> torch.Tensor:
        b = goal_xy.shape[0]
        c, s = np.cos(self.goal_z_rot), np.sin(self.goal_z_rot)
        g = torch.zeros(b, 3, 3, device=goal_xy.device)
        g[:, 0, 0], g[:, 0, 1], g[:, 1, 0], g[:, 1, 1] = c, -s, s, c
        g[:, 2, 2] = 1.0
        g[:, 0:2, 2] = goal_xy
        return torch.linalg.inv(g)

    def _initialize_episode(self, env_idx: torch.Tensor, options: dict):
        # Parent sets up the table, default goal/T and robot rest pose; we override all three.
        super()._initialize_episode(env_idx, options)
        with torch.device(self.device):
            b = len(env_idx)

            # ---- goal ----
            goal_xy = (options or {}).get("goal_xy")
            if goal_xy is None:
                goal_xy = sample_goals(b, self.goal_quadrants, self.device)
            else:
                goal_xy = torch.as_tensor(goal_xy, dtype=torch.float32, device=self.device).reshape(-1, 2)
                if goal_xy.shape[0] == 1 and b > 1:
                    goal_xy = goal_xy.expand(b, 2)
                elif goal_xy.shape[0] == self.num_envs and b < self.num_envs:
                    goal_xy = goal_xy[env_idx]
                assert goal_xy.shape[0] == b, "goal_xy must be [1, 2], [b, 2] or [num_envs, 2]"
            goal_p = torch.zeros(b, 3)
            goal_p[:, :2] = goal_xy
            goal_p[:, 2] = 1e-3
            goal_q = torch.tensor([np.cos(self.goal_z_rot / 2), 0.0, 0.0, np.sin(self.goal_z_rot / 2)],
                                  dtype=torch.float32).expand(b, 4)
            self.goal_tee.set_pose(Pose.create_from_pq(p=goal_p, q=goal_q))
            self.world_to_goal_trans[env_idx] = self._goal_world_to_goal(goal_xy)
            self.goal_quadrant[env_idx] = quadrant_of(goal_xy)

            # ---- T block start: independent of goal quadrant, distance-limited ----
            cand = _uniform_box((b, N_CANDIDATES), TEE_BOX, self.device)          # [b, K, 2]
            d = (cand - goal_xy[:, None, :]).norm(dim=-1)
            pick = _first_valid((d >= START_GOAL_DIST[0]) & (d <= START_GOAL_DIST[1]))
            tee_xy = cand[torch.arange(b), pick]
            tee_p = torch.zeros(b, 3)
            tee_p[:, :2] = tee_xy
            tee_p[:, 2] = 0.04 / 2 + 1e-3
            ang = torch.rand(b) * (2 * torch.pi)
            tee_q = torch.zeros(b, 4)
            tee_q[:, 0], tee_q[:, 3] = (ang / 2).cos(), (ang / 2).sin()
            self.tee.set_pose(Pose.create_from_pq(p=tee_p, q=tee_q))

            # ---- stick tip start: library pose away from the T ----
            n_lib = self.tip_xy_lib.shape[0]
            idx = torch.randint(n_lib, (b, N_CANDIDATES))
            dt = (self.tip_xy_lib[idx] - tee_xy[:, None, :]).norm(dim=-1)
            pick = _first_valid(dt >= TIP_TEE_MIN)
            qpos = self.tip_qpos_lib[idx[torch.arange(b), pick]]
            qpos = qpos + torch.randn_like(qpos) * self.robot_init_qpos_noise
            self.agent.reset(qpos)


In [ ]:
%%writefile /tmp/pusht_icl/repo/state_rl/pusht_state_ppo.py
"""Goal-conditioned state PPO teacher for PushTVarGoal-v1 (Jimmy).

Training itself is ManiSkill's unchanged PPO baseline (examples/baselines/ppo/ppo.py at
commit a4a4f92), launched through run_ppo.py so the variable-goal env is registered.
This module holds everything around it:

  preflight  GPU checks of the variable-goal env (partial resets, per-env success check),
             plus observation_contract.json
  eval       per-quadrant closed-loop evaluation of PPO checkpoints, deterministic mean
             action clipped to [-1, 1]; selects a checkpoint on validation seeds and
             reports whether it clears the freeze bar in every quadrant

Metrics follow state_bc: success_once (reached the goal at some step) and success_at_end
(at the goal at step 100); episodes run the full 100 steps with no early reset.
Evaluation seeds: episodes for quadrant q use seed (base_seed + 1000 * q).
"""
import argparse
import json
import sys
from pathlib import Path

import numpy as np
import torch
from torch import nn

sys.path.insert(0, str(Path(__file__).resolve().parent))
import pusht_vargoal as pv  # noqa: E402  (registers PushTVarGoal-v1)

ENV_ID = "PushTVarGoal-v1"
CONTROL_MODE = "pd_joint_delta_pos"
OBS_FIELDS = [("agent/qpos", 7), ("agent/qvel", 7), ("extra/tcp_pose", 7),
              ("extra/goal_pos", 3), ("extra/obj_pose", 7)]  # same order as state_bc


def make_env(num_envs, sim_backend="physx_cuda", quadrants=(1, 2, 3, 4), **kw):
    import gymnasium as gym
    return gym.make(ENV_ID, num_envs=num_envs, obs_mode="state", control_mode=CONTROL_MODE,
                    sim_backend=sim_backend, goal_quadrants=quadrants, **kw)


# --------------------------------------------------------------------------------------
# Policy: deterministic mean of ManiSkill's state PPO actor
# --------------------------------------------------------------------------------------

class PPOActor(nn.Module):
    """actor_mean of examples/baselines/ppo/ppo.py (3 x 256 tanh)."""

    def __init__(self, state_dim, action_dim):
        super().__init__()
        self.actor_mean = nn.Sequential(
            nn.Linear(state_dim, 256), nn.Tanh(), nn.Linear(256, 256), nn.Tanh(),
            nn.Linear(256, 256), nn.Tanh(), nn.Linear(256, action_dim))

    @torch.no_grad()
    def act(self, raw_state):
        return self.actor_mean(raw_state).clamp(-1.0, 1.0)


def load_actor(path, state_dim=31, action_dim=7, device="cpu"):
    weights = torch.load(path, map_location=device, weights_only=False)
    actor = PPOActor(state_dim, action_dim)
    actor.load_state_dict({k: v for k, v in weights.items() if k.startswith("actor_mean.")})
    return actor.to(device).eval()


# --------------------------------------------------------------------------------------
# Evaluation
# --------------------------------------------------------------------------------------

@torch.no_grad()
def rollout(env, actor, seed, max_steps=100):
    """One batch of episodes (one per env) for max_steps with no resets."""
    obs, _ = env.reset(seed=seed)
    n = obs.shape[0]
    dev = obs.device
    success_once = torch.zeros(n, dtype=torch.bool, device=dev)
    first = torch.full((n,), -1, dtype=torch.long, device=dev)
    success = success_once
    for t in range(max_steps):
        obs, _, _, _, info = env.step(actor.act(obs.float()))
        success = info["success"].bool()
        first = torch.where((first < 0) & success, torch.full_like(first, t + 1), first)
        success_once |= success
    steps = first[first > 0].float()
    return {"episodes": int(n), "seed": int(seed),
            "success_once": success_once.float().mean().item(),
            "success_at_end": success.float().mean().item(),
            "mean_steps_to_first_success": steps.mean().item() if len(steps) else None}


def eval_per_quadrant(env, actor, base_seed, quadrants=(1, 2, 3, 4), max_steps=100):
    u = env.unwrapped
    out = {}
    for q in quadrants:
        u.goal_quadrants = (q,)
        r = rollout(env, actor, base_seed + 1000 * q, max_steps)
        assert bool((u.goal_quadrant == q).all()), "goals left the requested quadrant"
        out[f"Q{q}"] = r
    u.goal_quadrants = (1, 2, 3, 4)
    so = [out[f"Q{q}"]["success_once"] for q in quadrants]
    se = [out[f"Q{q}"]["success_at_end"] for q in quadrants]
    out["worst_success_once"] = min(so)
    out["mean_success_once"] = float(np.mean(so))
    out["mean_success_at_end"] = float(np.mean(se))
    return out


def checkpoint_iteration(p: Path):
    s = p.stem.split("_")[-1]
    return int(s) if s.isdigit() else 10**9  # final_ckpt sorts last


def cmd_eval(args):
    device = torch.device(args.device)
    ckpts = sorted(Path(args.run_dir).glob("*.pt"), key=checkpoint_iteration) if args.run_dir else []
    ckpts = ckpts[-args.last:] if args.last else ckpts
    ckpts += [Path(c) for c in args.ckpt]
    assert ckpts, "no checkpoints found"
    env = make_env(args.episodes, sim_backend=args.sim_backend)
    out = Path(args.out)
    out.mkdir(parents=True, exist_ok=True)
    rows = []
    for c in ckpts:
        actor = load_actor(c, device=device)
        r = eval_per_quadrant(env, actor, args.val_seed)
        rows.append({"checkpoint": str(c), **r})
        print(f"{c.name:>18}  worst {r['worst_success_once']:.0%}  mean {r['mean_success_once']:.0%}  "
              + "  ".join(f"Q{q} {r[f'Q{q}']['success_once']:.0%}/{r[f'Q{q}']['success_at_end']:.0%}"
                          for q in (1, 2, 3, 4)), flush=True)
    (out / "val_results.json").write_text(json.dumps(rows, indent=2))
    # Selection on validation only: best worst-quadrant success_once, then mean, then later checkpoint.
    best = max(rows, key=lambda r: (r["worst_success_once"], r["mean_success_once"],
                                    checkpoint_iteration(Path(r["checkpoint"]))))
    print("\nSelected on validation:", best["checkpoint"])
    result = {"selected": best["checkpoint"], "val": best, "freeze_bar": args.bar,
              "val_seed": args.val_seed, "episodes_per_quadrant": args.episodes}
    if args.test_seed is not None:
        test = eval_per_quadrant(env, load_actor(best["checkpoint"], device=device), args.test_seed)
        result.update(test=test, test_seed=args.test_seed,
                      passes_bar=all(test[f"Q{q}"]["success_once"] >= args.bar for q in (1, 2, 3, 4)))
        print("Test (used once):  " + "  ".join(
            f"Q{q} {test[f'Q{q}']['success_once']:.0%}/{test[f'Q{q}']['success_at_end']:.0%}" for q in (1, 2, 3, 4)))
        print(f"Freeze bar {args.bar:.0%} success_once in every quadrant:",
              "PASS" if result["passes_bar"] else "FAIL")
    (out / "selection.json").write_text(json.dumps(result, indent=2))
    env.close()


# --------------------------------------------------------------------------------------
# Preflight
# --------------------------------------------------------------------------------------

def _place_tee(u, xy, ang):
    from mani_skill.utils.structs.pose import Pose
    b = xy.shape[0]
    p = torch.zeros(b, 3, device=xy.device)
    p[:, :2], p[:, 2] = xy, 0.021
    q = torch.zeros(b, 4, device=xy.device)
    q[:, 0], q[:, 3] = np.cos(ang / 2), np.sin(ang / 2)
    u.tee.set_pose(Pose.create_from_pq(p=p, q=q))


def preflight(num_envs=8, sim_backend="physx_cuda", out=None):
    env = make_env(num_envs, sim_backend=sim_backend)
    u = env.unwrapped
    obs, _ = env.reset(seed=0)
    from mani_skill.utils import gym_utils
    checks = {}

    # 1. observation layout and spaces match state_bc's contract
    raw = u._get_obs_agent()
    extra = u._get_obs_extra(u.get_info())
    parts = [raw["qpos"], raw["qvel"], extra["tcp_pose"], extra["goal_pos"], extra["obj_pose"]]
    checks["obs_layout"] = bool(torch.allclose(torch.cat(parts, -1).float(), obs.float(), atol=1e-5))
    checks["dims"] = tuple(obs.shape[1:]) == (31,) and tuple(u.single_action_space.shape) == (7,)
    checks["max_episode_steps_100"] = gym_utils.find_max_episode_steps_value(env) == 100

    # 2. every env scores success against its own goal
    goals = u.goal_tee.pose.p[:, :2].clone()
    _place_tee(u, goals, u.goal_z_rot)
    checks["success_at_own_goal"] = bool(u.evaluate()["success"].all())
    shifted = goals.roll(1, 0)
    far = (shifted - goals).norm(dim=1) > 0.04
    _place_tee(u, shifted, u.goal_z_rot)
    checks["no_success_at_other_goal"] = bool((~u.evaluate()["success"][far]).all()) if far.any() else True

    # 3. partial reset changes only the reset envs (what PPO's auto-reset does)
    if num_envs >= 2:
        env.reset(seed=1)
        before_goal = u.goal_tee.pose.p[:, :2].clone()
        before_trans = u.world_to_goal_trans.clone()
        idx = torch.arange(0, num_envs, 2, device=u.device)
        keep = torch.arange(1, num_envs, 2, device=u.device)
        env.reset(options=dict(env_idx=idx))
        after_goal = u.goal_tee.pose.p[:, :2]
        checks["partial_reset_keeps_others"] = bool(
            torch.allclose(after_goal[keep], before_goal[keep]) and
            torch.allclose(u.world_to_goal_trans[keep], before_trans[keep]))
        checks["partial_reset_changes_reset_envs"] = bool((after_goal[idx] - before_goal[idx]).norm(dim=1).gt(1e-4).all())
        _place_tee(u, after_goal.clone(), u.goal_z_rot)
        checks["success_after_partial_reset"] = bool(u.evaluate()["success"].all())
        checks["quadrant_labels_after_partial_reset"] = bool((u.goal_quadrant == pv.quadrant_of(after_goal)).all())

    # 4. quadrant restriction
    u.goal_quadrants = (4,)
    env.reset(seed=2)
    checks["quadrant_restriction"] = bool((u.goal_quadrant == 4).all())
    u.goal_quadrants = (1, 2, 3, 4)

    contract = {
        "env_id": ENV_ID, "base_env": "PushT-v1", "obs_mode": "state", "control_mode": CONTROL_MODE,
        "state_dim": 31, "action_dim": 7, "max_episode_steps": 100,
        "control_freq": u.control_freq, "sim_freq": u.sim_freq, "sim_backend": sim_backend,
        "state_fields": [{"name": n, "start": s, "stop": s + k}
                         for (n, k), s in zip(OBS_FIELDS, np.cumsum([0] + [k for _, k in OBS_FIELDS[:-1]]).tolist())],
        "action_low": u.single_action_space.low.tolist(), "action_high": u.single_action_space.high.tolist(),
        "goal_fields_to_hide_from_transformer": ["extra/goal_pos (indices 21:24)"],
        "workspace": {"quad_origin": pv.QUAD_ORIGIN, "goal_half": pv.GOAL_HALF, "axis_gap": pv.AXIS_GAP,
                      "tee_box": pv.TEE_BOX, "tip_box": pv.TIP_BOX, "start_goal_dist": pv.START_GOAL_DIST,
                      "tip_tee_min": pv.TIP_TEE_MIN, "goal_z_rot": float(u.goal_z_rot)},
        "preflight": checks,
    }
    if out:
        Path(out).mkdir(parents=True, exist_ok=True)
        (Path(out) / "observation_contract.json").write_text(json.dumps(contract, indent=2))
    env.close()
    for k, v in checks.items():
        print(f"{'PASS' if v else 'FAIL'}  {k}")
    if not all(checks.values()):
        raise SystemExit("Preflight failed")
    return contract


def main(argv=None):
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    sub = p.add_subparsers(dest="cmd", required=True)
    a = sub.add_parser("preflight")
    a.add_argument("--out", required=True)
    a.add_argument("--num-envs", type=int, default=8)
    a.add_argument("--sim-backend", default="physx_cuda")
    e = sub.add_parser("eval")
    e.add_argument("--run-dir", help="ppo.py run folder with ckpt_*.pt / final_ckpt.pt")
    e.add_argument("--ckpt", nargs="*", default=[])
    e.add_argument("--last", type=int, default=0, help="only the last N checkpoints of --run-dir")
    e.add_argument("--episodes", type=int, default=100, help="per quadrant")
    e.add_argument("--val-seed", type=int, default=1001)
    e.add_argument("--test-seed", type=int, default=None)
    e.add_argument("--bar", type=float, default=0.80)
    e.add_argument("--out", required=True)
    e.add_argument("--device", default="cuda" if torch.cuda.is_available() else "cpu")
    e.add_argument("--sim-backend", default="physx_cuda")
    args = p.parse_args(argv)
    if args.cmd == "preflight":
        preflight(args.num_envs, args.sim_backend, args.out)
    else:
        cmd_eval(args)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /tmp/pusht_icl/repo/state_icl/pusht_icl.py
"""In-context goal inference on variable-goal Push-T (Jimmy).

Follows CODEX_CONTEXT.md (variable-goal experiment) and PushT_ICL_Transformer_Spec.md (backbone and
token layout). The learner never sees the goal: a prompt holds demonstrations from ONE expert pushing to
ONE hidden goal, plus a query state from a separate episode with the same expert and goal; the label is the
expert's action at that state. One learner is trained on a balanced mixture of teachers (PPO and distilled
BC); the same frozen checkpoint is evaluated on every teacher.

Data conventions follow Nathan's state_bc/pusht_icl.py (same goal sets, demo seeds, eval seeds, file names and
fields), so demo files collected by either codebase can be used by the other.

Subcommands
  collect  roll out one frozen teacher on the fixed goals of every split -> <out>/<split>_<teacher>.npz
           (every teacher uses the same goals and reset seeds, so starts are matched across teachers)
  train    train on train-split prompts (Q1-Q3 goals) of the given teachers; select on val (Q1-Q3)
  eval     frozen evaluation per teacher: offline action error and closed-loop success with correct-goal,
           wrong-goal and no context, plus the expert itself (true goal) on the same starts

Splits (target positions; starts anywhere, including Q4):
  train Q1-Q3 | val Q1-Q3 (separate goals) | test_in Q1-Q3 (separate goals) | test_q4 Q4

Teachers are given as NAME=KIND:PATH, e.g. ppo=ppo:state_rl/teachers/x.pt, bc=bc:state_bc/teachers/y.pt.
"""
import argparse
import contextlib
import json
import sys
import time
from pathlib import Path

import numpy as np
import torch
from torch import nn

HERE = Path(__file__).resolve().parent
sys.path.insert(0, str(HERE.parent / "state_rl"))
import pusht_vargoal as pv  # noqa: E402  (registers PushTVarGoal-v1)
from pusht_state_ppo import load_actor, make_env  # noqa: E402

OBS_DIM = 31
GOAL_COLS = (21, 22, 23)                                     # extra/goal_pos, removed for the learner
STATE_COLS = tuple(i for i in range(OBS_DIM) if i not in GOAL_COLS)
STATE_FIELDS = ["agent/qpos[7]", "agent/qvel[7]", "extra/tcp_pose[7]", "extra/obj_pose[7]"]
ACT_DIM = 7
T_EP = 100

SPLITS = {   # same goal / demo / eval seeds as Nathan's state_bc/pusht_icl.py
    "train":   dict(quadrants=(1, 2, 3), goals=2048, goal_seed=11, demo_seed=1_000_000, eval_seed=7_100_000),
    "val":     dict(quadrants=(1, 2, 3), goals=128, goal_seed=12, demo_seed=2_000_000, eval_seed=7_200_000),
    "test_in": dict(quadrants=(1, 2, 3), goals=128, goal_seed=13, demo_seed=3_000_000, eval_seed=7_300_000),
    "test_q4": dict(quadrants=(4,), goals=128, goal_seed=14, demo_seed=4_000_000, eval_seed=7_400_000),
}


def snippet_steps(T, stride, device=None):
    """Steps kept from each support episode: every stride-th, always including the last (final T pose)."""
    return torch.arange(T - 1, -1, -stride, device=device).flip(0)


def autocast(device, enabled):
    if enabled and device.type == "cuda":
        return torch.autocast("cuda", dtype=torch.float16)
    return contextlib.nullcontext()


# ======================================================================================
# Teachers
# ======================================================================================

def parse_teacher(spec):
    """'name=kind:path' -> (name, kind, path)."""
    name, rest = spec.split("=", 1)
    kind, path = rest.split(":", 1)
    return name, kind, path


def load_teacher(kind, path, device):
    """Return an object with .act(obs [n, 31]) -> actions [n, 7] in [-1, 1] (deterministic)."""
    if kind == "ppo":
        return load_actor(path, device=device)
    if kind == "bc":
        import bc_teacher
        return bc_teacher.load(path, device=device)
    raise ValueError(f"unknown teacher kind {kind}")


# ======================================================================================
# Running jobs: (goal, episode) pairs in batches of the env's size
# ======================================================================================

@torch.no_grad()
def run_jobs(env, job_goals, policy, seed, record=False):
    """Run one 100-step episode per job. job_goals: [J, 2] (cpu). policy(obs, job_idx) -> actions.

    Jobs run in consecutive chunks of env.num_envs (the last chunk is padded, padding discarded). Chunk c
    resets with seed + c, so the same jobs, env size and seed give the same starts whatever the policy
    (ManiSkill seeds torch on reset; goals are given, not sampled).
    """
    u = env.unwrapped
    n = u.num_envs
    J = job_goals.shape[0]
    out = {k: [] for k in ("success_once", "first_success_step", "success_at_end", "final_overlap", "max_overlap",
                           "obs0")}
    if record:
        out["obs"], out["actions"] = [], []
    for c, start in enumerate(range(0, J, n)):
        idx = torch.arange(start, min(start + n, J))
        pad = torch.cat([idx, idx[-1:].expand(n - len(idx))])
        obs, _ = env.reset(seed=seed + c, options=dict(goal_xy=job_goals[pad].to(u.device)))
        assert torch.allclose(obs[:, 21:23].float().cpu(), job_goals[pad], atol=1e-5), "goal not applied"
        out["obs0"].append(obs[:len(idx)].float().cpu())
        so = torch.zeros(n, dtype=torch.bool, device=obs.device)
        first = torch.zeros(n, dtype=torch.long, device=obs.device)
        mx = torch.zeros(n, device=obs.device)
        O, A = [], []
        for t in range(T_EP):
            a = policy(obs.float(), pad.to(obs.device)).float().clamp(-1.0, 1.0)
            if record:
                O.append(obs.float()); A.append(a)
            obs, _, _, _, info = env.step(a)
            s = info["success"].bool()
            first = torch.where((first == 0) & s, torch.full_like(first, t + 1), first)
            so |= s
            ov = u.pseudo_render_intersection().float()
            mx = torch.maximum(mx, ov)
        keep = slice(0, len(idx))
        out["success_once"].append(so[keep].cpu()); out["success_at_end"].append(s[keep].cpu())
        out["first_success_step"].append(first[keep].cpu())
        out["final_overlap"].append(ov[keep].cpu()); out["max_overlap"].append(mx[keep].cpu())
        if record:
            out["obs"].append(torch.stack(O, 1)[keep].cpu()); out["actions"].append(torch.stack(A, 1)[keep].cpu())
    return {k: torch.cat(v) for k, v in out.items()}


# ======================================================================================
# collect
# ======================================================================================

def split_goals(n, quadrants, seed):
    """A split's goal set, reproducible from seed (Nathan's sample_goal_set): CPU, global RNG untouched."""
    with torch.random.fork_rng(devices=[]):
        torch.manual_seed(seed)
        return pv.sample_goals(n, quadrants).float().cpu()


def cmd_collect(args):
    name, kind, path = parse_teacher(args.teacher)
    out = Path(args.out)
    out.mkdir(parents=True, exist_ok=True)
    env = make_env(args.num_envs, sim_backend=args.sim_backend)
    teacher = load_teacher(kind, path, env.unwrapped.device)
    for split in args.splits:
        cfg = SPLITS[split]
        f = out / f"{split}_{name}.npz"
        if f.exists() and not args.overwrite:
            print("exists, skipping:", f, flush=True)
            continue
        n_goals = args.goals or cfg["goals"]
        goals = split_goals(n_goals, cfg["quadrants"], cfg["goal_seed"])
        E = args.episodes
        t0 = time.time()
        r = run_jobs(env, goals.repeat_interleave(E, 0), lambda o, j: teacher.act(o), seed=cfg["demo_seed"],
                     record=True)
        first = r["first_success_step"].reshape(n_goals, E).numpy()
        end = r["success_at_end"].reshape(n_goals, E).numpy()
        np.savez(f, obs=r["obs"].reshape(n_goals, E, T_EP, OBS_DIM).numpy().astype(np.float32),
                 actions=r["actions"].reshape(n_goals, E, T_EP, ACT_DIM).numpy().astype(np.float32),
                 first_success_step=first, success_at_end=end, goal_xy=goals.numpy(),
                 goal_quadrant=pv.quadrant_of(goals).numpy(),
                 final_overlap=r["final_overlap"].reshape(n_goals, E).numpy().astype(np.float32))
        meta = dict(split=split, teacher=name, kind=kind, goals=n_goals, episodes_per_goal=E, steps=T_EP,
                    env_id="PushTVarGoal-v1", quadrants=list(cfg["quadrants"]), goal_seed=cfg["goal_seed"],
                    demo_seed_base=cfg["demo_seed"], num_envs=args.num_envs, noise_std=0.0,
                    success_once=float((first > 0).mean()), success_at_end=float(end.mean()),
                    teacher_checkpoint=path, seconds=round(time.time() - t0, 1))
        f.with_suffix(".json").write_text(json.dumps(meta, indent=2))
        print(f"[{name}] {split:>7}: {n_goals} goals x {E} episodes  success-once {meta['success_once']:.1%}  "
              f"at end {meta['success_at_end']:.1%}  ({meta['seconds']:.0f}s)", flush=True)


# ======================================================================================
# Learner (PushT_ICL_Transformer_Spec.md sections 2, 4-6)
# ======================================================================================

class ICLPolicy(nn.Module):
    def __init__(self, d_s, d_a, H=1, n_positions=1024):
        super().__init__()
        from transformers import GPT2Config, GPT2Model
        self.d_s, self.d_a, self.H = d_s, d_a, H
        self.config = GPT2Config(
            vocab_size=1, n_positions=n_positions, n_embd=256, n_layer=12, n_head=8, n_inner=1024,
            activation_function="gelu_new", resid_pdrop=0.0, embd_pdrop=0.0, attn_pdrop=0.0,
            layer_norm_epsilon=1e-5, initializer_range=0.02, scale_attn_weights=True,
            scale_attn_by_inverse_layer_idx=False, reorder_and_upcast_attn=False,
            add_cross_attention=False, use_cache=False, bos_token_id=None, eos_token_id=None,
            attn_implementation="sdpa")   # fused causal attention; same math as eager
        self.backbone = GPT2Model(self.config)
        self.backbone.wte.weight.requires_grad_(False)
        self.state_encoder = nn.Linear(d_s, 256)
        self.action_encoder = nn.Linear(H * d_a, 256)
        self.action_head = nn.Linear(256, H * d_a)
        for layer in (self.state_encoder, self.action_encoder, self.action_head):
            nn.init.normal_(layer.weight, mean=0.0, std=0.02)
            nn.init.zeros_(layer.bias)

    def forward(self, support_states, support_actions, query_states):
        """support_states [B,N,d_s], support_actions [B,N,H*d_a] or [B,N,H,d_a], query_states [B,d_s]."""
        B, N = support_states.shape[:2]
        query = self.state_encoder(query_states).unsqueeze(1)
        if N:
            s = self.state_encoder(support_states)
            a = self.action_encoder(support_actions.reshape(B, N, -1))
            tokens = torch.cat((torch.stack((s, a), dim=2).flatten(1, 2), query), dim=1)
        else:
            tokens = query
        if tokens.shape[1] > self.config.n_positions:
            raise ValueError("Context exceeds the configured token budget")
        hidden = self.backbone(inputs_embeds=tokens).last_hidden_state
        return self.action_head(hidden[:, -1]).reshape(B, self.H, self.d_a)


def count_params(m):
    return sum(p.numel() for p in m.parameters()), sum(p.numel() for p in m.parameters() if p.requires_grad)


# ======================================================================================
# Episode banks and prompts (all on the device; no Python loop over examples)
# ======================================================================================

def compute_stats(paths):
    """State mean/std over the train splits of all teachers (one shared normalization)."""
    s, ss, n = 0.0, 0.0, 0
    for p in paths:
        x = np.load(p)["obs"][..., list(STATE_COLS)].reshape(-1, len(STATE_COLS)).astype(np.float64)
        s, ss, n = s + x.sum(0), ss + (x ** 2).sum(0), n + len(x)
    mean = s / n
    std = np.sqrt(np.maximum(ss / n - mean ** 2, 0.0))
    return {"mean": mean.tolist(), "std": np.maximum(std, 1e-6).tolist()}


class Bank:
    """One teacher's split on the device: states without goal fields (normalized), actions, success."""

    def __init__(self, path, device, stats):
        d = np.load(path)
        mean = np.asarray(stats["mean"], np.float32)
        std = np.asarray(stats["std"], np.float32)
        self.states = torch.as_tensor((d["obs"][..., list(STATE_COLS)] - mean) / std, device=device)
        self.actions = torch.as_tensor(d["actions"], device=device)    # already in [-1, 1]; not renormalized
        self.goal_xy = d["goal_xy"]
        self.goal_quadrant = d["goal_quadrant"]
        self.success_np = (d["first_success_step"] > 0) if "first_success_step" in d else d["success_once"]
        self.success = torch.as_tensor(self.success_np, device=device)
        self.n_good = self.success.sum(1)
        self.G, self.E, self.T = self.states.shape[:3]
        self.device = device

    def gather(self, g, sup, qe, qt, ts):
        """g [B], sup [B,k], qe [B], qt [B] (long tensors on device), ts [L] -> support/query tensors."""
        B, k = sup.shape
        L = len(ts)
        if k:
            gi = g[:, None, None].expand(B, k, L)
            ei = sup[:, :, None].expand(B, k, L)
            ti = ts[None, None, :].expand(B, k, L)
            ss = self.states[gi, ei, ti].reshape(B, k * L, -1)
            sa = self.actions[gi, ei, ti].reshape(B, k * L, -1)
        else:
            ss = self.states.new_zeros(B, 0, self.states.shape[-1])
            sa = self.actions.new_zeros(B, 0, self.actions.shape[-1])
        return ss, sa, self.states[g, qe, qt], self.actions[g, qe, qt]


def sample_prompts(bank, B, k, gen, ts):
    """Support: k distinct successful episodes of a random goal; query: a different episode, any step."""
    dev = bank.device
    eligible = torch.nonzero((bank.n_good >= k) & (bank.E > k)).squeeze(1)
    g = eligible[torch.randint(len(eligible), (B,), device=dev, generator=gen)]
    keys = torch.rand(B, bank.E, device=dev, generator=gen)
    keys = torch.where(bank.success[g], keys, keys + 2.0)            # successes sort first, in random order
    sup = keys.argsort(1)[:, :k]
    qkeys = torch.rand(B, bank.E, device=dev, generator=gen)
    if k:
        qkeys = qkeys.scatter(1, sup, 2.0)                          # never the support episodes
    qe = qkeys.argmin(1)
    qt = torch.randint(bank.T, (B,), device=dev, generator=gen)
    return bank.gather(g, sup, qe, qt, ts)


def eval_support_order(bank, n_support_eps, seed):
    """Per goal, a fixed order of episodes < n_support_eps: successful first (random order), then failures."""
    rng = np.random.default_rng(seed)
    order = []
    for g in range(bank.G):
        good = [e for e in rng.permutation(n_support_eps) if bank.success_np[g, e]]
        bad = [e for e in rng.permutation(n_support_eps) if not bank.success_np[g, e]]
        order.append(good + bad)
    return order


def wrong_goal_map(G):
    """Each goal's wrong-goal partner: a different goal of the same split (shift by G // 2)."""
    assert G >= 2, "wrong-goal context needs at least 2 goals in the split"
    return (np.arange(G) + max(1, G // 2)) % G


def context_for_goals(bank, goals, order, k, ts):
    """Support tensors [len(goals), k*L, .] from the first k episodes of each goal's order."""
    dev = bank.device
    sup = torch.as_tensor(np.array([order[g][:k] for g in goals], np.int64).reshape(len(goals), k), device=dev)
    gg = torch.as_tensor(np.asarray(goals, np.int64), device=dev)
    zero = torch.zeros(len(goals), dtype=torch.long, device=dev)
    ss, sa, _, _ = bank.gather(gg, sup, zero, zero, ts)
    return ss, sa


def failed_demo_fraction(bank, order, goals, k):
    """Fraction of context demos that are unsuccessful (only when a goal has fewer than k successes)."""
    if not k:
        return 0.0
    return float(np.mean([[not bank.success_np[g, e] for e in order[g][:k]] for g in goals]))


@torch.no_grad()
def offline_action_error(model, bank, ks, ts, n_support_eps, query_stride, amp, seed=0, chunk=1024):
    """Query-action MSE. Context: first k of each goal's support order (episodes < n_support_eps).
    Queries: every query_stride-th step of episodes >= n_support_eps. All goals batched together."""
    order = eval_support_order(bank, n_support_eps, seed)
    wrong = wrong_goal_map(bank.G)
    goals = list(range(bank.G))
    qsteps = torch.arange(0, bank.T, query_stride, device=bank.device)
    gg = torch.as_tensor(goals, device=bank.device)
    qs = bank.states[gg][:, n_support_eps:][:, :, qsteps].reshape(len(goals), -1, bank.states.shape[-1])
    qa = bank.actions[gg][:, n_support_eps:][:, :, qsteps].reshape(len(goals), -1, ACT_DIM)
    Q = qs.shape[1]
    owner = torch.arange(len(goals), device=bank.device).repeat_interleave(Q)
    qs, qa = qs.reshape(-1, qs.shape[-1]), qa.reshape(-1, ACT_DIM)
    res = {}
    for k in ks:
        for cond in (("correct", "wrong") if k else ("none",)):
            src = goals if cond != "wrong" else [int(wrong[g]) for g in goals]
            ss, sa = context_for_goals(bank, src, order, k, ts)
            se = 0.0
            for i in range(0, len(qs), chunk):
                o = owner[i:i + chunk]
                with autocast(bank.device, amp):
                    p = model(ss[o], sa[o], qs[i:i + chunk])[:, 0]
                se += (p.float() - qa[i:i + chunk]).pow(2).sum().item()
            res[f"{cond}_k{k}"] = {"mse": se / qa.numel(), "goals": len(goals), "queries": int(len(qs))}
    return res


# ======================================================================================
# train
# ======================================================================================

def cmd_train(args):
    torch.manual_seed(args.seed)
    dev = torch.device(args.device)
    amp = args.amp and dev.type == "cuda"
    data, out = Path(args.data), Path(args.out)
    out.mkdir(parents=True, exist_ok=True)
    stats = compute_stats([data / f"train_{t}.npz" for t in args.teachers])
    train = {t: Bank(data / f"train_{t}.npz", dev, stats) for t in args.teachers}
    val = {t: Bank(data / f"val_{t}.npz", dev, stats) for t in args.teachers}
    ts = snippet_steps(T_EP, args.stride, dev)
    assert 2 * max(args.ks) * len(ts) + 1 <= 1024, "context does not fit in 1024 tokens"
    model = ICLPolicy(len(STATE_COLS), ACT_DIM).to(dev)
    total, trainable = count_params(model)
    print(f"params total {total:,} trainable {trainable:,}  teachers {args.teachers}  amp {amp}", flush=True)
    opt = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=args.lr)
    scaler = torch.amp.GradScaler("cuda", enabled=amp)
    gen = torch.Generator(device=dev); gen.manual_seed(args.seed)
    rng = np.random.default_rng(args.seed)
    step, history, best = 0, [], None
    if args.resume:
        ck = torch.load(args.resume, map_location=dev, weights_only=False)
        model.load_state_dict(ck["model"]); opt.load_state_dict(ck["optimizer"]); scaler.load_state_dict(ck["scaler"])
        step, history, best = ck["step"], ck["history"], ck["best"]
        gen.manual_seed(args.seed + step); rng = np.random.default_rng(args.seed + step)
        print(f"resumed at step {step} (model + optimizer + scaler)", flush=True)
    import transformers
    config = {k: v for k, v in vars(args).items() if k != "func"} | {
        "state_cols": list(STATE_COLS), "state_fields": STATE_FIELDS, "removed_goal_cols": list(GOAL_COLS),
        "action": "pd_joint_delta_pos, 7-d, already in [-1, 1]; not renormalized; clipped at execution",
        "pairs_per_demo": len(ts), "params_total": total, "params_trainable": trainable,
        "torch": torch.__version__, "transformers": transformers.__version__,
        "data_meta": {f"{sp}_{t}": json.loads((data / f"{sp}_{t}.json").read_text())
                      for t in args.teachers for sp in ("train", "val") if (data / f"{sp}_{t}.json").exists()}}
    (out / "config.json").write_text(json.dumps(config, indent=2))
    (out / "state_stats.json").write_text(json.dumps(stats))

    def save(path, with_opt):
        blob = {"model": model.state_dict(), "step": step, "config": config, "stats": stats,
                "history": history, "best": best}
        if with_opt:
            blob |= {"optimizer": opt.state_dict(), "scaler": scaler.state_dict()}
        torch.save(blob, path)

    def validate():
        model.eval()
        v = {t: offline_action_error(model, val[t], args.ks, ts, args.n_support_eps, args.val_query_stride, amp)
             for t in args.teachers}
        model.train()
        score = float(np.mean([v[t][f"correct_k{k}"]["mse"] for t in args.teachers for k in args.select_ks]))
        return v, score

    teachers = list(args.teachers)
    start_step = step

    def lr_at(it):   # Nathan's schedule when --warmup/--lr-min-frac are set; constant otherwise
        if it < args.warmup:
            return args.lr * (it + 1) / args.warmup
        if args.lr_min_frac >= 1.0:
            return args.lr
        prog = min((it - args.warmup) / max(1, args.steps - args.warmup), 1.0)
        return args.lr * (args.lr_min_frac + (1 - args.lr_min_frac) * 0.5 * (1 + np.cos(np.pi * prog)))

    t0 = last_log = time.time()
    loss_sum, loss_n = torch.zeros((), device=dev), 0
    model.train()
    while step < args.steps:
        for group in opt.param_groups:
            group["lr"] = lr_at(step)
        k = int(rng.choice(args.ks))
        # teacher per example, uniform over teachers (balanced mixture regardless of dataset size)
        counts = np.bincount(rng.integers(len(teachers), size=args.batch), minlength=len(teachers))
        parts = [sample_prompts(train[t], int(c), k, gen, ts) for t, c in zip(teachers, counts) if c]
        ss, sa, qs, qa = (torch.cat(x) for x in zip(*parts))
        with autocast(dev, amp):
            pred = model(ss, sa, qs)[:, 0]
        loss = nn.functional.mse_loss(pred.float(), qa)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        nn.utils.clip_grad_norm_(model.parameters(), args.grad_clip)
        scaler.step(opt)
        scaler.update()
        step += 1
        loss_sum += loss.detach(); loss_n += 1
        if time.time() - last_log > 60 or step == args.steps:
            rate = (step - start_step) / (time.time() - t0)
            print(f"step {step}  loss {(loss_sum / loss_n).item():.4f}  lr {opt.param_groups[0]['lr']:.2e}  "
                  f"{rate:.1f} it/s  (ETA {(args.steps - step) / max(rate, 1e-9) / 60:.0f} min)", flush=True)
            last_log = time.time()
        if step % args.eval_every == 0 or step == args.steps:
            tv = time.time()
            v, score = validate()
            history.append({"step": step, "train_loss": (loss_sum / max(loss_n, 1)).item(), "val_select_mse": score,
                            **{f"{t}/{c}": r["mse"] for t in teachers for c, r in v[t].items()}})
            loss_sum, loss_n = torch.zeros((), device=dev), 0
            for t in teachers:
                print(f"[val {t}] step {step}  " + "  ".join(f"{c} {r['mse']:.4f}" for c, r in v[t].items()),
                      flush=True)
            print(f"[val] step {step}  select {score:.4f}  ({time.time() - tv:.0f}s)", flush=True)
            if best is None or score < best["val_select_mse"]:
                best = {"step": step, "val_select_mse": score}
                save(out / "best.pt", with_opt=False)
            save(out / "last.pt", with_opt=True)
            (out / "history.json").write_text(json.dumps(history, indent=2))
        if args.max_minutes and time.time() - t0 > 60 * args.max_minutes:
            print(f"time limit reached at step {step}", flush=True)
            save(out / "last.pt", with_opt=True)
            break
    print("best:", best, flush=True)


# ======================================================================================
# eval
# ======================================================================================

def load_learner(path, device):
    ck = torch.load(path, map_location=device, weights_only=False)
    model = ICLPolicy(len(STATE_COLS), ACT_DIM).to(device)
    model.load_state_dict(ck["model"])
    return model.eval(), ck


def summarize(r, quads):
    s = {m: r[m].float().mean().item() for m in ("success_once", "success_at_end", "final_overlap", "max_overlap")}
    s["episodes"] = int(len(r["success_once"]))
    for q in sorted(set(quads.tolist())):
        s[f"Q{q}_success_once"] = r["success_once"][quads == q].float().mean().item()
    return s


def cmd_eval(args):
    dev = torch.device(args.device)
    amp = args.amp and dev.type == "cuda"
    model, ck = load_learner(args.learner, dev)
    cfg = ck["config"]
    nsup, ks = cfg["n_support_eps"], args.ks
    ts = snippet_steps(T_EP, cfg["stride"], dev)
    env = make_env(args.num_envs, sim_backend=args.sim_backend)
    mean = torch.tensor(ck["stats"]["mean"], device=dev)
    std = torch.tensor(ck["stats"]["std"], device=dev)
    sc = torch.tensor(STATE_COLS, device=dev)
    out = Path(args.out)
    out.mkdir(parents=True, exist_ok=True)
    results = {"learner": str(args.learner), "learner_step": ck["step"], "trained_on": cfg["teachers"],
               "stride": cfg["stride"], "n_support_eps": nsup, "starts_per_goal": args.starts, "teachers": {}}

    def learner_policy(ss, sa):
        def act(obs, j):
            q = (obs.to(dev)[:, sc] - mean) / std
            with torch.no_grad(), autocast(dev, amp):
                return model(ss[j.to(dev)], sa[j.to(dev)], q)[:, 0].float().to(obs.device)
        return act

    for spec in args.teachers:
        name, kind, path = parse_teacher(spec)
        teacher = load_teacher(kind, path, env.unwrapped.device)
        results["teachers"][name] = {}
        for split in args.splits:
            bank = Bank(Path(args.data) / f"{split}_{name}.npz", dev, ck["stats"])
            order = eval_support_order(bank, nsup, seed=0)
            wrong = wrong_goal_map(bank.G)
            offline = offline_action_error(model, bank, ks, ts, nsup, args.query_stride, amp)
            goals = list(range(min(bank.G, args.goals or bank.G)))
            job_goal_idx = np.repeat(goals, args.starts)
            job_goals = torch.as_tensor(bank.goal_xy[job_goal_idx])
            quads = torch.as_tensor(bank.goal_quadrant[job_goal_idx])
            seed = SPLITS[split]["eval_seed"]                        # same starts for every teacher/condition
            rows = {}
            ref = run_jobs(env, job_goals, lambda o, j: teacher.act(o), seed)
            rows["expert_true_goal"] = summarize(ref, quads)
            for k in ks:
                for cond in (("correct", "wrong") if k else ("none",)):
                    src = job_goal_idx if cond != "wrong" else wrong[job_goal_idx]
                    ss, sa = context_for_goals(bank, src, order, k, ts)
                    r = run_jobs(env, job_goals, learner_policy(ss, sa), seed)
                    assert torch.allclose(r["obs0"], ref["obs0"], atol=1e-4), "starts differ between conditions"
                    s = rows[f"{cond}_k{k}"] = summarize(r, quads)
                    s["gap_to_expert_success_once"] = rows["expert_true_goal"]["success_once"] - s["success_once"]
                    print(f"[{name}] {split:>8} {cond:>7} k={k}  success {s['success_once']:.1%} "
                          f"(at end {s['success_at_end']:.1%})  overlap {s['final_overlap']:.2f}  "
                          f"action MSE {offline[f'{cond}_k{k}']['mse']:.4f}", flush=True)
            print(f"[{name}] {split:>8}  expert (true goal)  success {rows['expert_true_goal']['success_once']:.1%}",
                  flush=True)
            results["teachers"][name][split] = {
                "goals_evaluated": len(goals), "closed_loop": rows, "action_mse": offline,
                "failed_demo_fraction": {f"k{k}": failed_demo_fraction(bank, order, goals, k) for k in ks}}
            (out / "eval_results.json").write_text(json.dumps(results, indent=2))


# ======================================================================================

def main(argv=None):
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    sub = p.add_subparsers(dest="cmd", required=True)
    dev = "cuda" if torch.cuda.is_available() else "cpu"

    c = sub.add_parser("collect")
    c.add_argument("--teacher", required=True, help="name=kind:path")
    c.add_argument("--out", required=True)
    c.add_argument("--splits", nargs="+", default=list(SPLITS))
    c.add_argument("--episodes", type=int, default=10, help="episodes per goal")
    c.add_argument("--goals", type=int, default=0, help="goals per split (0 = the split's default; tests)")
    c.add_argument("--overwrite", action="store_true")
    c.add_argument("--num-envs", type=int, default=1024)
    c.add_argument("--sim-backend", default="physx_cuda")
    c.set_defaults(func=cmd_collect)

    t = sub.add_parser("train")
    t.add_argument("--data", required=True)
    t.add_argument("--teachers", nargs="+", required=True, help="teacher names: --data/train_<name>.npz")
    t.add_argument("--out", required=True)
    t.add_argument("--steps", type=int, default=50_000)
    t.add_argument("--batch", type=int, default=64)
    t.add_argument("--lr", type=float, default=1e-4)
    t.add_argument("--grad-clip", type=float, default=1.0)
    t.add_argument("--stride", type=int, default=4, help="keep every stride-th step of a demo")
    t.add_argument("--ks", type=int, nargs="+", default=[0, 1, 2, 4, 8], help="demos per prompt")
    t.add_argument("--select-ks", type=int, nargs="+", default=[1, 2, 4, 8])
    t.add_argument("--n-support-eps", type=int, default=8, help="eval: support pool = episodes < this")
    t.add_argument("--warmup", type=int, default=0, help="linear warmup steps (Nathan: 1000)")
    t.add_argument("--lr-min-frac", type=float, default=1.0, help="cosine decay to this fraction (Nathan: 0.1)")
    t.add_argument("--val-query-stride", type=int, default=10)
    t.add_argument("--eval-every", type=int, default=2000)
    t.add_argument("--max-minutes", type=float, default=0)
    t.add_argument("--seed", type=int, default=0)
    t.add_argument("--amp", type=int, default=1, help="fp16 autocast + grad scaling on CUDA")
    t.add_argument("--resume", default=None)
    t.add_argument("--device", default=dev)
    t.set_defaults(func=cmd_train)

    e = sub.add_parser("eval")
    e.add_argument("--learner", required=True)
    e.add_argument("--teachers", nargs="+", required=True, help="name=kind:path (data: <split>_<name>.npz)")
    e.add_argument("--data", required=True)
    e.add_argument("--out", required=True)
    e.add_argument("--splits", nargs="+", default=["val", "test_in", "test_q4"])
    e.add_argument("--goals", type=int, default=0, help="evaluate the first n goals of each split (0 = all)")
    e.add_argument("--ks", type=int, nargs="+", default=[0, 1, 2, 4, 8])
    e.add_argument("--starts", type=int, default=4, help="rollout starts per goal")
    e.add_argument("--query-stride", type=int, default=5, help="offline action error: every n-th query step")
    e.add_argument("--num-envs", type=int, default=512)
    e.add_argument("--sim-backend", default="physx_cuda")
    e.add_argument("--amp", type=int, default=1)
    e.add_argument("--device", default=dev)
    e.set_defaults(func=cmd_eval)

    args = p.parse_args(argv)
    args.func(args)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /tmp/pusht_icl/repo/state_icl/bc_teacher.py
"""Load Nathan's goal-conditioned BC teacher (state_bc run 4, distilled from the PPO teacher).

Copied from state_bc/pusht_state_bc.py (BCPolicy, load_policy) so state_icl does not import the whole BC
module. Checkpoint format: {"config": {...}, "state_dict": {...}, "extra": {...}}; the normalization
statistics are buffers in the state dict, so the policy takes raw 31-number observations (goal included).
"""
import numpy as np
import torch
from torch import nn


class BCPolicy(nn.Module):
    """MLP teacher: normalized state -> action in [-1, 1] (identical to state_bc's BCPolicy)."""

    def __init__(self, state_dim, action_dim, state_mean, state_scale, hidden=256, depth=2, tanh_output=True):
        super().__init__()
        layers, width = [], state_dim
        for _ in range(depth):
            layers += [nn.Linear(width, hidden), nn.ReLU()]
            width = hidden
        layers.append(nn.Linear(width, action_dim))
        self.net = nn.Sequential(*layers)
        self.tanh_output = tanh_output
        self.register_buffer("state_mean", torch.as_tensor(state_mean, dtype=torch.float32))
        self.register_buffer("state_scale", torch.as_tensor(state_scale, dtype=torch.float32))

    def forward(self, raw_state):
        out = self.net((raw_state - self.state_mean) / self.state_scale)
        return torch.tanh(out) if self.tanh_output else out

    @torch.no_grad()
    def act(self, raw_state):
        return self(raw_state).clamp(-1.0, 1.0)


def load(path, device="cpu"):
    ckpt = torch.load(path, map_location=device, weights_only=False)
    cfg = ckpt["config"]
    policy = BCPolicy(cfg["state_dim"], cfg["action_dim"], np.zeros(cfg["state_dim"]), np.ones(cfg["state_dim"]),
                      hidden=cfg["hidden"], depth=cfg["depth"], tanh_output=cfg["tanh_output"])
    policy.load_state_dict(ckpt["state_dict"])
    assert cfg["state_dim"] == 31 and cfg["action_dim"] == 7, "expected the 31-number goal-conditioned state"
    return policy.to(device).eval()


In [ ]:
BLOBS = [
("state_rl/assets/tip_start_qpos.npz", "..."),
("state_rl/teachers/ppo_vargoal_seed1_50M_actor.pt", "..."),
]

for rel, sha, b64 in BLOBS:
    data = base64.b64decode(b64)
    assert hashlib.sha256(data).hexdigest() == sha, rel
    (SRC / rel).write_bytes(data)
shutil.copytree(SRC, OUTPUT_ROOT / "code", dirs_exist_ok=True)
ICL = SRC / "state_icl/pusht_icl.py"
SPECS = {"ppo": f"ppo=ppo:{SRC / 'state_rl/teachers/ppo_vargoal_seed1_50M_actor.pt'}",
         "bc": f"bc=bc:{SRC / 'state_icl/teachers/bc_teacher.pt'}"}
TEACHER_SPECS = [SPECS[t] for t in TEACHERS]
run([PY, ICL, "--help"])

## 3. Smoke test on the GPU (tiny collect, 20 training steps, tiny eval)

In [ ]:
SMOKE = WORK / "smoke"
shutil.rmtree(SMOKE, ignore_errors=True)
for spec in TEACHER_SPECS:
    run([PY, ICL, "collect", "--teacher", spec, "--out", SMOKE / "data", "--episodes", 4, "--goals", 6, "--num-envs", 24])
run([PY, ICL, "train", "--data", SMOKE / "data", "--teachers", *TEACHERS, "--out", SMOKE / "run", "--steps", 20,
     "--eval-every", 10, "--batch", 16, "--ks", 0, 1, 2, "--select-ks", 1, 2, "--n-support-eps", 3])
run([PY, ICL, "eval", "--learner", SMOKE / "run/best.pt", "--teachers", *TEACHER_SPECS, "--data", SMOKE / "data",
     "--out", SMOKE / "eval", "--ks", 0, 1, "--starts", 2, "--num-envs", 12, "--splits", "test_q4"])

## 4. Teacher demonstrations: reuse the attached cache, or collect

To reuse demos from an earlier run: Add Input → Notebook Output Files → this notebook (or the collection run).

In [ ]:
cached = sorted(Path("/kaggle/input").glob("**/train_ppo.npz")) if Path("/kaggle/input").exists() else []
DATA = OUTPUT_ROOT / "data"
DATA.mkdir(exist_ok=True)
if cached:
    for f in cached[0].parent.glob("*_*.*"):                # npz + json of every split and teacher
        (DATA / f.name).symlink_to(f)
    print("Using cached demos from", cached[0].parent)
for spec in TEACHER_SPECS:                                  # collects only what is missing
    run([PY, ICL, "collect", "--teacher", spec, "--out", DATA, "--episodes", 10, "--num-envs", 1024])
for f in sorted(DATA.glob("*.json")):
    m = json.loads(f.read_text())
    print(f"{f.stem:>14}: {m['goals']} goals x {m['episodes_per_goal']}  success-once {m['success_once']:.1%}  "
          f"at end {m['success_at_end']:.1%}")

## 5. Train two seeds at once (one per T4)

In [ ]:
RUNS = {seed: OUTPUT_ROOT / f"run_seed{seed}" for seed in SEEDS}
run_parallel({f"train_seed{seed}": ([PY, ICL, "train", "--data", DATA, "--teachers", *TEACHERS, "--out", RUNS[seed],
                                     "--steps", TRAIN_STEPS, "--eval-every", EVAL_EVERY, "--warmup", WARMUP,
                                     "--lr-min-frac", LR_MIN_FRAC, "--max-minutes", MAX_MINUTES, "--seed", seed], gpu)
              for gpu, seed in enumerate(SEEDS)})

## 6. Frozen evaluation of both seeds (one per T4)

In [ ]:
run_parallel({f"eval_seed{seed}": ([PY, ICL, "eval", "--learner", RUNS[seed] / "best.pt", "--teachers", *TEACHER_SPECS,
                                    "--data", DATA, "--out", RUNS[seed] / "eval"], gpu)
              for gpu, seed in enumerate(SEEDS)})
for seed in SEEDS:
    res = json.loads((RUNS[seed] / "eval" / "eval_results.json").read_text())
    print(f"\n=== seed {seed} (step {res['learner_step']}) ===")
    for teacher, splits in res["teachers"].items():
        for split, r in splits.items():
            print(f"[{teacher}] {split}  ({r['goals_evaluated']} goals x {res['starts_per_goal']} starts)")
            for cond, x in r["closed_loop"].items():
                mse = r["action_mse"].get(cond, {}).get("mse")
                print(f"  {cond:>17}  success {x['success_once']:6.1%}  at end {x['success_at_end']:6.1%}  "
                      f"overlap {x['final_overlap']:.2f}" + (f"  action MSE {mse:.4f}" if mse is not None else ""))

## 7. Save

Outputs persist in the committed version: demos (`data/`, reusable as a cache), both runs, evaluations.

In [ ]:
for f in DATA.iterdir():                                    # store cached demos as real files in this output
    if f.is_symlink():
        target = f.resolve(); f.unlink(); shutil.copy2(target, f)
print("Folder:", OUTPUT_ROOT)